In [2]:
import pandas as pd
from tqdm.std import tqdm
from openai import OpenAI
import re
import os
import json

In [3]:
input_data = "input_files/semantic_matching_0109.csv"
output_data = "output_files/semantic_matching_0109.xlsx"

In [4]:
from dotenv import load_dotenv

load_dotenv()

True

## common functions

In [5]:
def to_pascal_case(phrase):
    # Remove parentheses and the content inside them, e.g., (Unaudited)
    phrase = re.sub(r'\s*\([^)]*\)', '', phrase)
    
    # Split on non-alphanumeric characters
    words = re.split(r'[^a-zA-Z0-9]', phrase)
    
    # Capitalize each word and join
    return ''.join(word.capitalize() for word in words if word)

In [6]:
def read_xml(xml_path):
    with open(xml_path, "r", encoding="utf-8") as f:
        content = f.read()

        return content

In [7]:
"""
    This function is different a bit among different tasks, here is for the semantic matching task
"""

def construct_annotation(df, results, task_name="semantic_matching"):
    """
        the options of task_name are semantic_matching, relationship_extraction, and mathematical_reasoning
    """
    new_df = df.copy(deep=True)

    schema = []
    presentation = []
    calculation = []
    definition = []
    label = []
    instance = []
    us_gaap = []
    true_answer = []
    segmentation_path = []
    parse_ok = []
    input = []
    outputByLLM = []
    
    for res in results:
        folder_name = res.get("folder_name")

        prefix_path = os.path.join(f"segmentation_filing/{task_name}", folder_name)
        real_segmentation_names = os.listdir(prefix_path)

        
        parse = False
        cal = None
        pre = None
        defn = None
        lab = None
        ins = None
        sch = None
        seg_path = None
        answers = [tag.get("dimension") for tag in res.get("error_tags")]
        for segmentation_name_ in real_segmentation_names:
            
            full_path = os.path.join(prefix_path, segmentation_name_)
    
            temp_cal = None
            temp_pre = None
            temp_defn = None
            temp_lab = None
            temp_ins = None
            temp_sch = None
            
            file_list = os.listdir(full_path)
            
            for file_name in file_list:
                if file_name.endswith("cal.xml"):
                    final_path = os.path.join(full_path,file_name)
                    temp_cal = read_xml(final_path)
                elif file_name.endswith("pre.xml"):
                    final_path = os.path.join(full_path,file_name)
                    temp_pre = read_xml(final_path)
                elif file_name.endswith("def.xml"):
                    final_path = os.path.join(full_path,file_name)
                    temp_defn = read_xml(final_path)
                elif file_name.endswith("lab.xml"):
                    final_path = os.path.join(full_path,file_name)
                    temp_lab = read_xml(final_path)
                elif file_name.endswith("htm.xml"):
                    final_path = os.path.join(full_path,file_name)
                    temp_ins = read_xml(final_path)
                elif file_name.endswith("xsd"):
                    final_path = os.path.join(full_path,file_name)
                    temp_sch = read_xml(final_path)
                else:
                    continue

            is_appear = True

            ans = "us-gaap:ConcentrationRiskPercentage1"
            # check the instance document
            if ans in temp_ins:
                pass
            elif ans.replace(":","") in temp_ins:
                pass
            elif ans.replace(":","_") in temp_ins:
                pass
            else:
                is_appear = False
           
            if is_appear:
                parse = True
                cal = temp_cal
                pre = temp_pre
                defn = temp_defn
                lab = temp_lab
                ins = temp_ins
                sch = temp_sch
                seg_path = os.path.join(folder_name, segmentation_name_)
                break
        
        calculation.append(cal)
        presentation.append(pre)
        definition.append(defn)
        label.append(lab)
        instance.append(ins)
        schema.append(sch)
        true_answer.append(answers)
        us_gaap.append(None)
        parse_ok.append(parse)
        segmentation_path.append(seg_path)
        input.append(None)
        outputByLLM.append(res)
     
        
    new_df["schema"] = schema
    new_df["presentation"] = presentation
    new_df["calculation"] = calculation
    new_df["definition"] = definition
    new_df["label"] = label
    new_df["instance"] = instance
    new_df["us_gaap"] = us_gaap
    new_df["input"] = input
    new_df["true_answer"] = true_answer
    new_df["segmentation_path"] = segmentation_path
    new_df["parse_successful"] = parse_ok
    new_df["LLM_Output"] = outputByLLM

    return new_df

## load dqc messages for semantic matching task

In [8]:
sm_109_df = pd.read_csv(input_data)

In [9]:
sm_109_df

,assertion.code,assertion.detail,assertion.run-date,assertion.severity,assertion.source,assertion.type,entity.cik,entity.name,report.accepted-timestamp,report.accession,report.base-taxonomy,report.creation-software,report.document-type,report.entry-url,report.filing-date,report.filing-year,report.id,report.sec-url,report.sic-code,folder_name
0,DQC.US.0109.9569,"2022-03-09 20:08:42,685 [DQC.US.0109.9569] The...",2022-03-10,ERROR,DQC,109,788920,PRO DEX INC,2022-02-03 16:01:00,0001553350-22-000085,US GAAP 2021,Field: Set; Name: xdx; ID: xdx_05E_edei%2D%2DC...,10-Q,http://www.sec.gov/Archives/edgar/data/788920/...,2022-02-03,2022,415967,https://www.sec.gov/Archives/edgar/data/788920...,3841,10q-pdex-20211231
1,DQC.US.0109.9569,"2022-03-09 20:08:42,686 [DQC.US.0109.9569] The...",2022-03-10,ERROR,DQC,109,788920,PRO DEX INC,2022-02-03 16:01:00,0001553350-22-000085,US GAAP 2021,Field: Set; Name: xdx; ID: xdx_05E_edei%2D%2DC...,10-Q,http://www.sec.gov/Archives/edgar/data/788920/...,2022-02-03,2022,415967,https://www.sec.gov/Archives/edgar/data/788920...,3841,10q-pdex-20211231
2,DQC.US.0109.9569,"2022-03-09 20:08:42,686 [DQC.US.0109.9569] The...",2022-03-10,ERROR,DQC,109,788920,PRO DEX INC,2022-02-03 16:01:00,0001553350-22-000085,US GAAP 2021,Field: Set; Name: xdx; ID: xdx_05E_edei%2D%2DC...,10-Q,http://www.sec.gov/Archives/edgar/data/788920/...,2022-02-03,2022,415967,https://www.sec.gov/Archives/edgar/data/788920...,3841,10q-pdex-20211231
3,DQC.US.0109.9569,"2022-03-09 20:08:42,687 [DQC.US.0109.9569] The...",2022-03-10,ERROR,DQC,109,788920,PRO DEX INC,2022-02-03 16:01:00,0001553350-22-000085,US GAAP 2021,Field: Set; Name: xdx; ID: xdx_05E_edei%2D%2DC...,10-Q,http://www.sec.gov/Archives/edgar/data/788920/...,2022-02-03,2022,415967,https://www.sec.gov/Archives/edgar/data/788920...,3841,10q-pdex-20211231
4,DQC.US.0109.9569,"2022-03-09 20:08:42,687 [DQC.US.0109.9569] The...",2022-03-10,ERROR,DQC,109,788920,PRO DEX INC,2022-02-03 16:01:00,0001553350-22-000085,US GAAP 2021,Field: Set; Name: xdx; ID: xdx_05E_edei%2D%2DC...,10-Q,http://www.sec.gov/Archives/edgar/data/788920/...,2022-02-03,2022,415967,https://www.sec.gov/Archives/edgar/data/788920...,3841,10q-pdex-20211231
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
153,DQC.US.0109.9569,"2024-10-11 18:03:25,832 [DQC.US.0109.9569] The...",2024-10-11,ERROR,DQC,109,885462,"Gulf Resources, Inc.",2024-10-11 16:31:00,0001193805-24-001241,US GAAP 2024,Novaworks,10-Q,http://www.sec.gov/Archives/edgar/data/885462/...,2024-10-11,2024,767919,https://www.sec.gov/Archives/edgar/data/885462...,2800,10q-gure-20240630
154,DQC.US.0109.9569,"2024-10-11 18:03:21,788 [DQC.US.0109.9569] The...",2024-10-11,ERROR,DQC,109,885462,"Gulf Resources, Inc.",2024-10-11 16:30:00,0001193805-24-001240,US GAAP 2024,Novaworks,10-Q,http://www.sec.gov/Archives/edgar/data/885462/...,2024-10-11,2024,767920,https://www.sec.gov/Archives/edgar/data/885462...,2800,10q-gure-20240331
155,DQC.US.0109.9569,"2024-10-11 18:03:21,788 [DQC.US.0109.9569] The...",2024-10-11,ERROR,DQC,109,885462,"Gulf Resources, Inc.",2024-10-11 16:30:00,0001193805-24-001240,US GAAP 2024,Novaworks,10-Q,http://www.sec.gov/Archives/edgar/data/885462/...,2024-10-11,2024,767920,https://www.sec.gov/Archives/edgar/data/885462...,2800,10q-gure-20240331
156,DQC.US.0109.9569,"2024-09-27 19:19:29,973 [DQC.US.0109.9569] The...",2024-09-27,ERROR,DQC,109,885462,"Gulf Resources, Inc.",2024-09-27 16:20:00,0001193805-24-001184,US GAAP 2023,Novaworks,10-K,http://www.sec.gov/Archives/edgar/data/885462/...,2024-09-27,2024,765432,https://www.sec.gov/Archives/edgar/data/885462...,2800,10k-gure-20231231


## LLM judge assisstant

In [10]:
Client = OpenAI()

In [11]:
sm_109_system_prompt = """You are given a DQC validation message. Your task is to extract all Dimensions information from the message.

1. Extract all key-value pairs under Dimensions, where each dimension is formatted as us-gaap:XXX=YYY.

2. For each extracted dimension, provide a short explanation in the "reason" field based on the message context. If no explicit issue is mentioned about a specific dimension, you may leave the "reason" as an empty string.

Output your result as a structured JSON in the following format:
```json
{
  "error_tags": [
    {
      "dimension": "...",
      "reason": "..."
    }
  ]
}

"""

In [12]:
def get_response(dqc_message):
    
    completion = Client.chat.completions.create(
        model="gpt-4.1-mini",
        response_format={
            "type": "json_object"
        },
        messages=[
            {"role": "system", "content": sm_109_system_prompt},
            {"role": "user", "content": f"DQC validation message: {dqc_message} \n Output:"}
        ]
    )
    return completion.choices[0].message.content

In [ ]:
res = []
for _, row in tqdm(sm_109_df.iterrows()):
    ## load each dqc message
    message = row["assertion.detail"]
    response = get_response(message)
    response = json.loads(response)

    ## load the folder name which will be used to obtain candidate segmentation folders
    folder_name = row["folder_name"]

    ## record the folder name
    response["folder_name"] = folder_name
    res.append(response)

6it [00:15,  2.52s/it]

In [ ]:
res[0]

{'error_tags': [{'dimension': 'us-gaap:ConcentrationRiskByTypeAxis=pdex:CustomerConcentrationRisk2Member',
   'reason': 'The fact needs another axis to identify the specific pdex:CustomerConcentrationRisk2Member risk.'},
  {'dimension': 'us-gaap:ConcentrationRiskByBenchmarkAxis=us-gaap:SalesMember',
   'reason': 'The concept us-gaap:ConcentrationRiskPercentage1 should always be used with the ConcentrationRiskByBenchmarkAxis.'}],
 'folder_name': '10q-pdex-20211231'}

In [ ]:
"""
    the options of task_name are semantic_matching, relationship_extraction, and mathematical_reasoning
"""
new_sm_109_df = construct_annotation(sm_109_df, res, task_name="semantic_matching")

In [ ]:
new_sm_109_df

,assertion.code,assertion.detail,assertion.run-date,assertion.severity,assertion.source,assertion.type,entity.cik,entity.name,report.accepted-timestamp,report.accession,...,calculation,definition,label,instance,us_gaap,input,true_answer,segmentation_path,parse_successful,LLM_Output
0,DQC.US.0109.9569,"2022-03-09 20:08:42,685 [DQC.US.0109.9569] The...",2022-03-10,ERROR,DQC,109,788920,PRO DEX INC,2022-02-03 16:01:00,0001553350-22-000085,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<link:...,<?xml version='1.0' encoding='UTF-8'?>\n<xbrl ...,None,None,[us-gaap:ConcentrationRiskByTypeAxis=pdex:Cust...,10q-pdex-20211231/MajorCustomersAndSuppliersDe...,True,{'error_tags': [{'dimension': 'us-gaap:Concent...
1,DQC.US.0109.9569,"2022-03-09 20:08:42,686 [DQC.US.0109.9569] The...",2022-03-10,ERROR,DQC,109,788920,PRO DEX INC,2022-02-03 16:01:00,0001553350-22-000085,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<link:...,<?xml version='1.0' encoding='UTF-8'?>\n<xbrl ...,None,None,[us-gaap:ConcentrationRiskByTypeAxis=pdex:Cust...,10q-pdex-20211231/MajorCustomersAndSuppliersDe...,True,{'error_tags': [{'dimension': 'us-gaap:Concent...
2,DQC.US.0109.9569,"2022-03-09 20:08:42,686 [DQC.US.0109.9569] The...",2022-03-10,ERROR,DQC,109,788920,PRO DEX INC,2022-02-03 16:01:00,0001553350-22-000085,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<link:...,<?xml version='1.0' encoding='UTF-8'?>\n<xbrl ...,None,None,[us-gaap:ConcentrationRiskByTypeAxis=us-gaap:C...,10q-pdex-20211231/MajorCustomersAndSuppliersDe...,True,{'error_tags': [{'dimension': 'us-gaap:Concent...
3,DQC.US.0109.9569,"2022-03-09 20:08:42,687 [DQC.US.0109.9569] The...",2022-03-10,ERROR,DQC,109,788920,PRO DEX INC,2022-02-03 16:01:00,0001553350-22-000085,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<link:...,<?xml version='1.0' encoding='UTF-8'?>\n<xbrl ...,None,None,[us-gaap:ConcentrationRiskByBenchmarkAxis=us-g...,10q-pdex-20211231/MajorCustomersAndSuppliersDe...,True,{'error_tags': [{'dimension': 'us-gaap:Concent...
4,DQC.US.0109.9569,"2022-03-09 20:08:42,687 [DQC.US.0109.9569] The...",2022-03-10,ERROR,DQC,109,788920,PRO DEX INC,2022-02-03 16:01:00,0001553350-22-000085,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<link:...,<?xml version='1.0' encoding='UTF-8'?>\n<xbrl ...,None,None,[us-gaap:ConcentrationRiskByBenchmarkAxis=us-g...,10q-pdex-20211231/MajorCustomersAndSuppliersDe...,True,{'error_tags': [{'dimension': 'us-gaap:Concent...
5,DQC.US.0109.9569,"2022-03-09 20:08:42,687 [DQC.US.0109.9569] The...",2022-03-10,ERROR,DQC,109,788920,PRO DEX INC,2022-02-03 16:01:00,0001553350-22-000085,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<link:...,<?xml version='1.0' encoding='UTF-8'?>\n<xbrl ...,None,None,[us-gaap:ConcentrationRiskByBenchmarkAxis=us-g...,10q-pdex-20211231/MajorCustomersAndSuppliersDe...,True,{'error_tags': [{'dimension': 'us-gaap:Concent...
6,DQC.US.0109.9569,"2022-03-09 20:08:42,688 [DQC.US.0109.9569] The...",2022-03-10,ERROR,DQC,109,788920,PRO DEX INC,2022-02-03 16:01:00,0001553350-22-000085,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<link:...,<?xml version='1.0' encoding='UTF-8'?>\n<xbrl ...,None,None,[us-gaap:ConcentrationRiskByBenchmarkAxis=us-g...,10q-pdex-20211231/MajorCustomersAndSuppliersDe...,True,{'error_tags': [{'dimension': 'us-gaap:Concent...
7,DQC.US.0109.9569,"2022-03-09 20:08:42,68

In [ ]:
new_sm_109_df.to_excel(output_data,index=False)